In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib import gridspec
from scipy.integrate import quad, dblquad
import time

In [ ]:
def fs(x,v0,L):
    return v0*(1+np.sin((4*np.pi*x)/10.0))

In [ ]:
def fs_grad(x,v0,L):
    return ((4*v0*np.pi)/10.0)*np.cos((4*np.pi*x)/10.0)

In [ ]:
def chain(N):
    con_mat = np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            if i==j:
                if i==0 or i==N-1:
                    con_mat[i][j] = 1
                else:
                    con_mat[i][j] = 2
            elif j==i-1 or j==i+1:
                con_mat[i][j] = -1
    return con_mat

def ring(N):
    con_mat_ring = np.zeros((N,N))
    con_mat_ring[0][N-1] = -1
    con_mat_ring[N-1][0] = -1
    for i in range(N):
        for j in range(N):
            if i==j:
                con_mat_ring[i][j] = 2
            elif j==i-1 or j==i+1:
                con_mat_ring[i][j] = -1
    return con_mat_ring

def star(N):
    con_mat_star = np.zeros((N,N))
    con_mat_star[0][0] = N-1
    for i in range(1,N):
        con_mat_star[i][i] = 1
        con_mat_star[0][i] = -1
        con_mat_star[i][0] = -1
    return con_mat_star

def clique(N):
    con_mat_clique = np.zeros((N,N))
    for i in range(N):
        con_mat_clique[i][i] = N-1
        for j in range(1,N):
            if i!=j:
                con_mat_clique[i][j] = -1
                con_mat_clique[j][i] = -1
    return con_mat_clique

In [ ]:
def Veff(x):
    return (1.0-epsilon/2.0)*(tau/2.0)*2.0*fs(x,v0,L)*fs_grad(x,v0,L) - vw*N


def Deff(x):
    return Dt + (tau/2.0)*((fs(x,v0,L))**2)

def b_func(x):
    return ((1.0-epsilon/2.0)*(tau/2.0)*2.0*fs(x,v0,L)*fs_grad(x,v0,L) - vw*N)/(Dt + (tau/2.0)*((fs(x,v0,L))**2))

    
def steady_state_density(x):
    def density_inner_int(y):
        g1 = x
        g2 = x+y
        return np.exp(-quad(b_func,g1,g2)[0])
    return (1/(Dt + (tau/2.0)*((fs(x,v0,L))**2)))*quad(density_inner_int,0,L)[0]



def outer_int():
    "integrates over x from 0 to L"
    def middle_int(x):
        "integrates over y from 0 to L to give a function of x"
        def inner_int(y):
            "integrates over y' from x to x+y to give a function of x and y"
            h1 = x
            h2 = x+y
            return np.exp(-quad(b_func,h1,h2)[0])
        return (1/(Dt + (tau/2.0)*((fs(x,v0,L))**2)))*quad(inner_int,0,L)[0]
    return quad(middle_int,0,L,limit=60)[0]

def drift():
    return 1-np.exp(-quad(b_func,0,L)[0])

def density_pol(x):
    return (1 + (tau/(2*Dt))*(fs(x,v0,L)**2))**(-(.5*epsilon))

In [ ]:
%%time

L = 10.
gamma = 1.
T = 0.1
Dr = 5.0
tau = 1/Dr
k = 5.0
mu = 1.0
v0 = 1.0
Dt = T
D = T
vw = 0.01
N = 6
xx = np.linspace(0,L,499)

tmp2 = []
tmp2 = np.vstack([xx])

fig, axs = plt.subplots(2, 1, sharex=True,height_ratios=[4,1])
fig.subplots_adjust(hspace=0)

for c in range(4):
    if c == 0:
        chain(N)
        evalue,evec=np.linalg.eig(chain(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*evalue[i])

        epsilon = 2.0 - summ
        print(epsilon)

    if c == 1:
        ring(N)
        evalue,evec=np.linalg.eig(ring(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*evalue[i])

        epsilon = 2.0 - summ
        print(epsilon)

    if c == 2:
        star(N)
        evalue,evec=np.linalg.eig(star(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*np.real(evalue[i]))

        epsilon = 2.0 - summ
        print(epsilon)

    if c == 3:
        clique(N)
        evalue,evec=np.linalg.eig(clique(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*evalue[i])

        epsilon = 2.0 - summ
        print(epsilon)


    norm_pol = L/(quad(density_pol,0.0,L)[0])
    cons1 = outer_int()

    axs[0].plot(xx,[L*steady_state_density(i)/(cons1) for i in xx],label='Numerical, N = %.2f' %c)
    tmp2 = np.vstack([tmp2,[L*steady_state_density(i)/(cons1) for i in xx]])
    axs[0].set_ylabel(r"$\frac{\rho(x)}{\rho_b}$",fontsize=18,labelpad=20,rotation='horizontal')
    axs[0].set_xlabel(r"$x$")
    axs[0].legend(loc="best", fontsize=10, frameon=True)

    axs[1].plot(xx,fs(xx,v0,L),'k-')
    axs[1].set_xlabel(r"$x$",fontsize=18)
    axs[1].set_ylabel(r"$f_s(x)$",fontsize=15,labelpad=20,rotation='horizontal')

In [ ]:
steps = 40
vw_i = 0.001
vw_f = 0.5
logx = np.zeros(steps+1)
stepsize = np.log10(vw_f/vw_i)/(steps-1)
for i in range(0,steps,1):
    logx[i+1] = 10**(np.log10(vw_i)+stepsize*i)

print(logx)

In [ ]:
%%time

L = 10.
gamma = 1.
T = 0.1
Dr = 5.
tau = 1/Dr
k = 5.0
mu = 1.0
v0 = 1.0
Dt = T
D = T
N = 6
xx = np.linspace(0,L,999)
drift_array=np.zeros(len(logx))

tmp2_test = []
tmp2_test = np.vstack([logx])

for c in range(4):
    if c == 0:
        chain(N)
        evalue,evec=np.linalg.eig(chain(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*evalue[i])

        epsilon = 2.0 - summ
        print(epsilon)

    if c == 1:
        ring(N)
        evalue,evec=np.linalg.eig(ring(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*evalue[i])

        epsilon = 2.0 - summ
        print(epsilon)

    if c == 2:
        star(N)
        evalue,evec=np.linalg.eig(star(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*np.real(evalue[i]))

        epsilon = 2.0 - summ
        print(epsilon)

    if c == 3:
        clique(N)
        evalue,evec=np.linalg.eig(clique(N))
        summ=0
        for i in range(len(evalue)):
            if evalue[i]<10**(-9):
                evalue[i]=0
            summ += 1/(1+tau*k*mu*evalue[i])

        epsilon = 2.0 - summ
        print(epsilon)

    for j in range(len(logx)):
        vw=logx[j]
        cons1 = outer_int()
        cons2 = drift()
        drift_array[j]=((L*cons2)/cons1)/N+vw
    tmp2_test = np.vstack([tmp2_test,[drift_array]])   
    plt.semilogx(logx,drift_array,label='N = %.2f' %N)
    plt.ylabel(r"$\frac{v_d}{v_0}$",fontsize=18,labelpad=20,rotation='horizontal')
    plt.xlabel(r"$v_w/v_0$")
    plt.legend(loc="best", fontsize=10, frameon=True)

In [ ]:
fig, axs = plt.subplots(2, 1, sharex=True,height_ratios=[4,1])
fig.subplots_adjust(hspace=0)

xx = np.linspace(0.0,L,1001)

    
axs[0].plot(A1[:,0],A1[:,1],'r',marker = 'o', mfc = 'none',ls='')
axs[0].plot(tmp2[0,:],tmp2[1,:],'lightcoral')
axs[0].plot(A2[:,0],A2[:,1],'b',marker = '^', mfc = 'none',ls='')
axs[0].plot(tmp2[0,:],tmp2[2,:],'cornflowerblue')
axs[0].plot(A3[:,0],A3[:,1],'g',marker = 's', mfc = 'none',ls='') 
axs[0].plot(tmp2[0,:],tmp2[3,:],'forestgreen')
axs[0].plot(A4[:,0],A4[:,1],'rebeccapurple',marker = '*',markersize=9, mfc = 'none', ls='')
axs[0].plot(tmp2[0,:],tmp2[4,:],'mediumpurple')
red_line = mlines.Line2D([], [], color='red', marker='o',mfc='none',ls='-', label='chain')
blue_line = mlines.Line2D([], [], color='blue', marker='^',mfc='none',ls='-', label='ring')
green_line = mlines.Line2D([], [], color='green', marker='s',mfc='none',ls='-', label='star')
purple_line = mlines.Line2D([], [], color='rebeccapurple',marker='*',markersize=9,mfc='none',ls='-', label='clique')
axs[0].legend(loc="best", fontsize=10, frameon=True,handles=[red_line,blue_line,green_line,purple_line])
axs[0].set_ylabel(r"$\frac{\rho(x)}{\rho_b}$",fontsize=18,labelpad=20,rotation='horizontal')
axs[0].set_xlim(left=0.0,right=10.0)

axs[1].plot(xx,fs(xx,v0,L),'k-')
axs[1].set_xlabel(r"$x$",fontsize=18)
axs[1].set_ylabel(r"$f_s(x)$",fontsize=15,labelpad=20,rotation='horizontal')

#plt.savefig('density_waves_topology.pdf')